# Teaching a small AI how an expert thinks ✏️
### … by adjusting **5,000 numbers** out of **500,000,000**

Modern AI language models are huge grids of numbers ("parameters"). Making a model better at a
specific job — *fine-tuning* — normally means re-training **all** of those numbers, which needs
big GPUs, lots of time, and lots of money.

**NTK fine-tuning** takes a different route: it leaves the model completely untouched and learns a
tiny *controller* — a set of just **5,000 dials** — that gently re-balances how the model thinks.

| Method | Numbers trained | Hardware to train | Result |
|---|---|---|---|
| Full fine-tune | ~494,000,000 | multi-GPU server, hours | new full copy of the model |
| LoRA (popular shortcut) | ~1–4,000,000 | 1 GPU | adapter file |
| **NTK controller (this demo)** | **5,000** | **a laptop-class CPU works** | **35 KB controller file** |

Today's demo: we taught **Qwen2.5-0.5B** (a small open model) the *reasoning style* of a
grade-school math tutor, using only **64 solved example problems** and **240 training steps** —
and we can measure exactly how much better it understands the tutor's way of solving problems.


## Setup

The demo talks to a live model endpoint running on our platform (deployed with one API call —
shown at the end). The "before" answers were captured from the very same platform serving the
original model, so the comparison is apples-to-apples.


In [ ]:
import json, re, textwrap
from pathlib import Path

import requests

# Live NTK-tuned endpoint (in-cluster service DNS; no auth needed pod-to-pod).
NTK_ENDPOINT = "http://ntk-gsm8k-predictor.admin.svc.cluster.local/openai/v1"

RESULTS_DIR = Path("demo_results")
base = json.loads((RESULTS_DIR / "base.json").read_text())
ntk = json.loads((RESULTS_DIR / "ntk.json").read_text())

ANSWER_RE = re.compile(r"####\s*([-+]?[\d,]*\.?\d+)")
NUMBER_RE = re.compile(r"[-+]?[\d,]*\.?\d+")

def extract_answer(text):
    m = ANSWER_RE.search(text)
    raw = m.group(1) if m else (NUMBER_RE.findall(text) or [None])[-1]
    if raw is None:
        return None
    raw = raw.replace(",", "").rstrip(".")
    try:
        v = float(raw)
    except ValueError:
        return None
    return int(v) if v == int(v) else v

def served_model_name(endpoint):
    r = requests.get(endpoint.rstrip("/") + "/models", timeout=30)
    r.raise_for_status()
    return r.json()["data"][0]["id"]

def ask_live(endpoint, model, question_prompt, max_tokens=256):
    body = {"model": model, "prompt": question_prompt, "temperature": 0,
            "max_tokens": max_tokens, "stop": ["Question:"]}
    r = requests.post(endpoint.rstrip("/") + "/completions", json=body, timeout=180)
    r.raise_for_status()
    return r.json()["choices"][0]["text"]

def show(title, question, answer_text, gold, pred):
    ok = pred is not None and pred == gold
    mark = "✅ CORRECT" if ok else "❌ WRONG"
    q = question.removeprefix("Question:").removesuffix("Answer:").strip()
    print("=" * 78)
    print(f"{title}   →   {mark}   (its answer: {pred}, correct answer: {gold})")
    print("-" * 78)
    print(textwrap.fill("Q: " + q, width=78))
    print()
    print(textwrap.indent(answer_text.strip()[:600], "   "))
    print()

# Showcase items: prefer problems the original model got wrong and the tuned one got right.
flips = [i for i, (b, n) in enumerate(zip(base["items"], ntk["items"]))
         if not b["correct"] and n["correct"]]
SHOWCASE = (flips or [i for i, b in enumerate(base["items"]) if not b["correct"]] or [0, 1, 2])[:4]
print(f"Loaded {base['n_items']} test problems. Showcase items: {SHOWCASE}")


## 1 · BEFORE — the original model tries these problems

These are real answers the **unmodified** model gave (captured from the platform, greedy decoding —
the model's single most-confident answer, no sampling tricks). Notice *how* it answers, not just
the final number: it rambles, invents follow-up questions, and drifts away from the task.


In [ ]:
for i in SHOWCASE:
    it = base["items"][i]
    show("ORIGINAL MODEL", it["question"], it["generation"], it["gold_answer"], it["pred_answer"])


## 2 · The fine-tune

We gave the platform **64 solved examples** and pressed go. The NTK trainer ran **240 steps** and
produced a **35 KB controller** (5,000 dials). The model's 500 million weights were never touched —
at serving time the controller rides along and re-balances the model's internal signals with exact
math (no approximation).

*This is running live on the cluster right now — the next cell asks the tuned model the very same
questions, in real time.*


In [ ]:
NTK_MODEL = served_model_name(NTK_ENDPOINT)
print(f"live endpoint model: {NTK_MODEL}\n")
for i in SHOWCASE:
    it = ntk["items"][i]
    live = ask_live(NTK_ENDPOINT, NTK_MODEL, it["question"])
    pred = extract_answer(live)
    show("NTK-TUNED MODEL (LIVE)", it["question"], live, it["gold_answer"], pred)


## 3 · The measurement — how well does it understand the expert now?

We measure *prediction quality* on 32 held-out problems the model **never saw in training**: show
each model the expert's correct solution, word by word, and record how "surprised" it is at every
step. A model that has internalized the expert's method is barely surprised; a model that hasn't
keeps guessing wrong. This is the standard scientific measure (perplexity) — and it's exactly the
metric the NTK method's published results are based on.


In [ ]:
import math
import matplotlib.pyplot as plt

labels = ["Original\nmodel", "NTK-tuned\n(+5,000 dials)"]
colors = ["#9aa5b1", "#2e7d32"]
ppl = [base.get("perplexity") or math.exp(base["nll"]), ntk.get("perplexity") or math.exp(ntk["nll"])]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
bars = ax1.bar(labels, ppl, color=colors)
ax1.set_ylabel("surprise per word (lower is better)")
ax1.set_title("How surprised is the model by expert solutions?")
for b, v in zip(bars, ppl):
    ax1.text(b.get_x() + b.get_width() / 2, v * 1.005, f"{v:.2f}", ha="center", fontweight="bold")

impr = (1 - ntk["nll"] / base["nll"]) * 100
sizes = [500_000_000, 5_000]
ax2.bar(["model parameters\n(untouched)", "NTK dials\n(trained)"], sizes, color=["#9aa5b1", "#2e7d32"])
ax2.set_yscale("log")
ax2.set_ylabel("count (log scale)")
ax2.set_title("What we had to train to get there")
ax2.text(1, 5_000 * 1.4, "5,000", ha="center", fontweight="bold")
ax2.text(0, 500_000_000 * 1.4, "500,000,000", ha="center", fontweight="bold")

fig.suptitle(f"{impr:.0f}% better prediction of expert solutions — from training 0.001% of the numbers",
             fontweight="bold")
fig.tight_layout()
plt.show()

print(f"Prediction quality (negative log-likelihood): {base['nll']:.4f} -> {ntk['nll']:.4f} "
      f"({impr:.0f}% improvement), matching the method's published benchmark behavior.")


## 4 · How it was done — one API call on the platform

The whole fine-tune is a single request to the platform. It validates the base model and dataset,
runs the training pipeline, stores the controller in the model catalog, and the result can be
served with one more call.

```json
POST /cogapi/models/fine-tune
{
  "base_model_id": "8f7318ab-6e4e-4e94-aeb9-6a61b6dd51e6",   // Qwen2.5-0.5B-Instruct
  "dataset_id":    "a6693b90-b11a-4af1-a31b-51cd3c1a84e8",   // 64 solved math problems
  "output_name":   "math-tutor-v1",
  "method": "ntk",
  "export": "ntk_model"
}
```

Serving the result (what powers the live endpoint you just saw):

```json
POST /cogapi/models-serving
{
  "isvc_name": "ntk-gsm8k",
  "model_id":  "8f7318ab-6e4e-4e94-aeb9-6a61b6dd51e6",
  "llm_adapter": { "kind": "ntk_model",
                   "adapter_model_id": "1a62543f-52f7-4871-82a7-f25a6dcd33f5" }
}
```

**Why this matters:** fine-tuning stops being a big-budget project. A 35 KB controller per use-case
means dozens of specializations of one base model — trained in minutes, stored for pennies, served
with exact math on our standard GPU runtime.


---
## Presenter appendix (ops — not part of the audience demo)

- **Endpoint check before going on stage:** run the Setup cell + `served_model_name(NTK_ENDPOINT)`.
  If it fails, redeploy with the `POST /models-serving` body above (header `kubeflow-userid: admin@hiro.com`,
  dev CogAPI `http://cog-api-dev.kubeflow/apidev`) and wait for ISVC `ntk-gsm8k` Ready in ns `admin`.
- **GPU juggling:** the L40 fits one model. Demo state = `ntk-gsm8k` up, qwen38 down.
  - free GPU: `kubectl scale deploy qwen38-predictor -n admin --replicas=0`
  - restore after demo: delete ISVC `ntk-gsm8k` via CogAPI, then `kubectl scale deploy qwen38-predictor -n admin --replicas=1`
- **Reference numbers (§F benchmark, 2026-06):** base NLL 0.7706 · NTK served NLL 0.5895 ·
  hook-reference 0.5886 (0.15% gap ⇒ the served controller is exact, not a silent no-op).
  Controller sha256 `52d9ea00…`; runtime `hiroregistry/cfhfserver:0.18.11-gpu` (needs ≥ 0.18.9).
- `demo_results/*.json` were captured with `scripts/demo_eval.py` (repo `cf_finetune_ntk`).
- **If asked about solve-rate/accuracy:** on the 32-problem set final-answer accuracy is statistically
  unchanged (8/32 base vs 5/32 tuned — noise at this sample size). NTK re-balances what the model
  already knows (measured by the 23% NLL gain, the method's published metric); raw arithmetic ability
  is limited by the 0.5B base model. The levers for accuracy are a larger base model and more
  training data — the NTK cost advantage is unchanged in both cases. Don't present accuracy claims;
  the showcase items are problems where the improvement is visible, not a claim about the average.
